# 06 · Household master table

Builds `data/processed/household_master.parquet`: one row per EICV7 household, with welfare, demographics, livelihoods, expenditure by price cell, housing, assets, services, savings and credit, and VUP.

| | |
|---|---|
| **Inputs** | EICV7 files in `Microdata/Cross_Section/` (read only); deduplicated section 9 files from notebook 05 in `data/interim/`; `reports/data_inspection/expenditure_cpi_mapping.csv` |
| **Outputs** | `data/processed/household_master.parquet`, `docs/household_master_data_dictionary.md`, `data/interim/s8b_food_item_values.parquet` (food diary cache) |
| **Code** | `src/features/build_household_master.py` |
| **Report** | `reports/household_master_summary.md` |
| **Runtime** | About 1 minute (the 1.1 GB food diary is read once and cached) |

**Rules.** One row per household. `weight` for household estimates, `pop_wt` for person estimates. NISR values are kept unchanged; derived values are new columns. FinScope is not used. Section 9 totals are missing for the households notebook 05 could not deduplicate.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "features" / "build_household_master.py").exists())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from IPython.display import display

from src.features import build_household_master as bm

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", "{:,.3f}".format)

def wmean(x, w):
    ok = x.notna()
    return np.average(x[ok].astype(float), weights=w[ok])

## 1. How the expenditure split reconciles with NISR's aggregates

The CPI shock (notebook 07) needs each household's spending by price category. NISR publishes 11 aggregates per household (`exp1`–`exp11`) that sum to consumption (`cons1`). The master table keeps those totals unchanged and splits each one across price cells, in proportion to the household's own item-level values.

The split is only valid if the item data are what NISR summed. The table below compares each aggregate with the household's item total.

In [ ]:
mapping = bm.load_mapping()
items = bm.item_values(mapping)
pov = bm.read("poverty").set_index("hhid")
rows = []
for agg, it in items.items():
    tot = it.groupby("hhid")["value"].sum()
    r = (pov[agg] / tot).replace([np.inf, -np.inf], np.nan).dropna()
    r = r[pov.loc[r.index, agg] > 0]
    rows.append({"aggregate": agg, "households with items": len(r),
                 "ratio p1": r.quantile(0.01), "p25": r.quantile(0.25), "median": r.median(), "p75": r.quantile(0.75), "p99": r.quantile(0.99),
                 "households with aggregate > 0 but no items": int(((pov[agg] > 0) & ~pov.index.isin(tot[tot > 0].index)).sum())})
recon = pd.DataFrame(rows).set_index("aggregate")
recon

How to read the ratios (aggregate ÷ item total):

- **`exp6` = 1.000 for every household.** The annual non-food aggregate equals S8A1 purchases plus gifts, once 11 item codes are left out: diagnostic tests, hospitalisation, giving birth, emergency transport, assistive devices, dwelling repair materials, security equipment, mattresses, irons and musical instruments. NISR keeps these lumpy or durable purchases out of consumption, so the split leaves them out too.
- **`exp7` = 13.04 = 365 ÷ 28.** Monthly items (4-week recall) annualised.
- **`exp8` = 52.14 = 365 ÷ 7.** Weekly items, including own production, annualised.
- **`exp9` ≥ 52.14.** The diary covers 7 days. NISR values consumed quantities at prices we cannot reproduce exactly, so the ratio varies. The split uses the household's own quantities valued at its own unit values (else the area median), and only the proportions matter.
- **`exp1` ≤ 1.** NISR's education aggregate equals the person-level components (fees, contributions, uniforms, books, transport, other, plus food at school) for most households. It leaves out room and board and vocational training.

Households with an aggregate above zero but no item detail take the weighted item mix of their area (urban or rural). This affects very few households; see `n_aggregates_split_by_area_average`.

In [ ]:
food = bm.food_item_values()
print(f"Food diary cache: {len(food):,} household-item rows; {food['hhid'].nunique():,} households")
parts = food[["value_purchased", "value_own", "value_gift"]].sum()
print("Value by source (share of diary food value):", (parts / parts.sum()).round(3).to_dict())
print(f"Item rows with a quantity but no price: {int(food['unpriced_rows'].sum())}")

## 2. Build the table and run the checks

In [ ]:
master = bm.build()
checks = bm.checks(master)
assert checks["result"].eq("pass").all(), checks[checks["result"] != "pass"]
print(f"{len(master):,} rows x {master.shape[1]} columns")
checks

## 3. Poverty

The poverty rate must reproduce NISR's 27.4% when weighted by `pop_wt`. Weighting by the household weight `weight` gives the share of **households** that are poor, which is lower because poor households are larger.

In [ ]:
pw, w = master["pop_wt"], master["weight"]
rates = pd.DataFrame({
    "poor, % of people (pop_wt)": master.groupby("area").apply(lambda g: wmean(g["poor"], g["pop_wt"]) * 100),
    "extreme poor, % of people (pop_wt)": master.groupby("area").apply(lambda g: wmean(g["extreme_poor"], g["pop_wt"]) * 100),
    "poor, % of households (weight)": master.groupby("area").apply(lambda g: wmean(g["poor"], g["weight"]) * 100),
})
rates.loc["Rwanda"] = [wmean(master["poor"], pw) * 100, wmean(master["extreme_poor"], pw) * 100, wmean(master["poor"], w) * 100]
rates

In [ ]:
by_prov = master.groupby("province_name").apply(lambda g: pd.Series({
    "poor, % of people": wmean(g["poor"], g["pop_wt"]) * 100,
    "extreme poor, % of people": wmean(g["extreme_poor"], g["pop_wt"]) * 100,
    "sample households": len(g)}))
by_prov.sort_values("poor, % of people")

## 4. District coverage

Every district must appear, with enough households in each district and urban/rural cell to support estimates.

In [ ]:
cov = master.groupby(["province_name", "district_name", "area"]).agg(households=("hhid", "size"), weighted_households=("weight", "sum")).unstack("area")
cov.columns = [f"{a} {b}" for a, b in cov.columns]
cov = cov.fillna(0)
print(f"Districts: {master['district'].nunique()} | district x area cells: {master.groupby(['district', 'ur']).ngroups} | strata: {master['strata_id'].nunique()}")
print(f"Sample households per district: min {master.groupby('district').size().min()}, max {master.groupby('district').size().max()}")
print(f"Smallest district x area cells:")
display(master.groupby(["district_name", "area"]).size().nsmallest(5).rename("households").to_frame())
cov

## 5. VUP participation

Weighted shares of households. `vup_any` is built from the five component flags. It does not use `s9d1q1`, which despite its label only flags each file's own component. Asset transfers and skills development, also part of VUP, are not in the public cross-section files.

In [ ]:
vup_cols = ["vup_direct_support", "vup_classic_pw", "vup_expanded_pw", "vup_nsds", "vup_financial_services", "vup_any"]
vtab = pd.DataFrame({
    "sample households": master[vup_cols].sum(),
    "% of households (weight)": [wmean(master[c], w) * 100 for c in vup_cols],
    "% of poor households": [wmean(master.loc[master["poor"].eq(1), c], w[master["poor"].eq(1)]) * 100 for c in vup_cols],
    "% of non-poor households": [wmean(master.loc[master["poor"].eq(0), c], w[master["poor"].eq(0)]) * 100 for c in vup_cols],
    "% of people (pop_wt)": [wmean(master[c], pw) * 100 for c in vup_cols],
})
display(vtab)
print("Households by number of components:", master["vup_n_components"].value_counts().sort_index().to_dict())
amounts = ["vup_ds_amount_12m", "vup_cpw_amount_12m", "vup_epw_amount_12m", "vup_nsds_quarterly_payment", "vup_nsds_amount_12m_est", "vup_fs_loan_amount"]
display(master[amounts].describe(percentiles=[0.5]).T[["count", "50%", "mean"]].rename(columns={"50%": "median RWF", "mean": "mean RWF"}))
print(f"Public Works beneficiaries without a payment total: classic {int(master['vup_cpw_amount_missing'].sum())}, expanded {int(master['vup_epw_amount_missing'].sum())} (flagged, not imputed)")

## 6. Key indicators by area

Weighted household means. Shares are fractions of `cons_nominal`.

In [ ]:
show = ["share_food", "share_food_nisr", "share_meals_away", "share_housing_utilities", "share_housing_utilities_excl_imputed_rent",
        "share_energy", "share_transport", "share_education", "share_education_nisr", "share_communication", "share_food_own_produced",
        "hh_size", "dependency_ratio", "head_female", "head_age", "agriculture_dependent", "share_employed_informal_proxy",
        "any_wage_worker", "improved_drinking_water", "electricity_any", "clean_cooking_fuel", "basic_sanitation", "owns_mobile_phone",
        "any_savings_instrument", "any_formal_loan", "any_remittance_received", "any_public_support"]
ind = pd.DataFrame({a: {c: wmean(g[c], g["weight"]) for c in show} for a, g in [("Rwanda", master), *master.groupby("area")]})
ind

In [ ]:
liv = (master.groupby(["main_livelihood", "area"])["weight"].sum().unstack("area")
       .pipe(lambda t: t / t.sum() * 100).round(1))
liv["poverty rate, % of people"] = master.groupby("main_livelihood").apply(lambda g: wmean(g["poor"], g["pop_wt"]) * 100)
liv

## 7. Missingness

Most missing values are structural: a question that does not apply (for example, a payment amount for non-beneficiaries). The table lists every column with any missing value.

In [ ]:
dd = bm.data_dictionary(master)
miss = dd[dd["pct_missing"] > 0][["column", "block", "pct_missing"]].sort_values("pct_missing", ascending=False)
print(f"{len(miss)} of {len(dd)} columns have missing values; {int((dd['pct_missing'] == 0).sum())} are complete.")
miss

## 8. Save

In [ ]:
path = bm.save(master)
dict_path = bm.write_data_dictionary(master, ROOT / "docs" / "household_master_data_dictionary.md")
check = pd.read_parquet(path)
assert check.shape == master.shape and check["hhid"].is_unique
print(f"{path.relative_to(ROOT)}: {check.shape[0]:,} rows x {check.shape[1]} columns, {path.stat().st_size / 1e6:.1f} MB")
print(f"{dict_path.relative_to(ROOT)}: {len(dd)} columns documented")